In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import xarray as xr
import pandas as pd
import icechunk
import tools
from icechunk.xarray import to_icechunk
from dask.distributed import Client
from evaltools.source import get_source_collection, open_and_sort, open_datasets

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/WCRP-CORDEX/data-request-table/refs/heads/main/data-request/dreq_default.csv")
core_variables = df[(df.priority == "CORE") & (df.frequency == "mon")].out_name.tolist()
core_variables

In [ ]:
client = Client(dashboard_address="localhost:8787", threads_per_worker=1)

In [ ]:
catalog = get_source_collection(
    project_id="CORDEX-CMIP6",
    variable_id=core_variables,
    frequency="mon",
    add_fx=["orog", "sftlf", "sftlaf", "areacella"],
    driving_experiment_id="evaluation",
)
dsets = open_and_sort(catalog, merge_fx=True, apply_fixes=True)

In [ ]:
for key, ds in dsets.items():
    dsets[key] = tools.preprocess_dataset(ds)

## Write to S3 without icechunk

In [ ]:
import cordex as cx

def rechunk(ds, refvar=None):
    """Naive rechunking along time"""
    if refvar is None:
        refvar = [var for var in ds.data_vars if "time" in ds[var].dims][0]
    chunksize = int(ds[refvar].sizes["time"] * 100 * 1024**2 / ds[refvar].nbytes)
    chunks = {dim: chunksize if dim == "time" else -1 for dim in ds.dims}
    print(f"Rechunking to {chunks}")
    return ds.chunk(chunks)


def write_to_s3(key, ds):
    #target = fsspec.get_mapper(f"s3://euro-cordex/CORDEX-CMIP6/{key}")
    target = f"s3://euro-cordex/CORDEX-CMIP6"
    rechunk(ds).to_zarr(target, zarr_format=3, compute=True, group=key)

In [ ]:
# run zarr conversion

for key, ds in dsets.items():
    print(f"Writing {key} to s3...")
    print(ds.cf["grid_mapping"].grid_mapping_name)
    print(ds.dims)
    write_to_s3(key, ds)

## Open with xarray and obstore

In [ ]:
import os
import xarray as xr
from zarr.storage import ObjectStore
from obstore.store import S3Store

from boto3 import Session
from obstore.auth.boto3 import Boto3CredentialProvider

# Choose the dataset prefix you want to open
#prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120"
prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.mon.v20240920/"
#prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.CESAM-UA.ERA5.evaluation.r1i1p1f1.WRF451Q.v1-r2.mon.v20250630/"
#prefix = "CMIP5/cordex/output/EUR-11/GERICS/ECMWF-ERAINT/evaluation/r1i1p1/REMO2015/v1/mon/tas/v20180813/"
prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.mon.v20240920/"
#prefix = "CORDEX-CMIP6"

# Use ~/.aws/credentials via AWS_PROFILE (defaults to "default") and region
profile = os.environ.get("AWS_PROFILE", "default")
region = os.environ.get("AWS_REGION", "eu-central-1")

# Create a boto3 Session using your local credentials and region
session = Session(profile_name=profile, region_name=region)
credential_provider = Boto3CredentialProvider(session)

# Build an authenticated S3Store (do NOT set skip_signature when using credentials)
s3_store = S3Store("euro-cordex", credential_provider=credential_provider, region=region, prefix=prefix)
#s3_store = S3Store("euro-cordex", skip_signature=True, region=region, prefix=prefix)

# Wrap in a zarr ObjectStore and open with xarray
store = ObjectStore(s3_store, read_only=True)

ds = xr.open_dataset(store, consolidated=False, engine="zarr", decode_coords="all")
ds

## Virtualization with virtualizarr

In [3]:
from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import ZarrParser
from obstore.store import from_url
from obstore.store import S3Store
from boto3 import Session
from obstore.auth.boto3 import Boto3CredentialProvider
import os 

import nest_asyncio
nest_asyncio.apply()

bucket = "euro-cordex"
# Dataset prefix (root of the Zarr store). Must match where zarr.json lives.
#prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/tas"
prefix = "CMIP5/cordex/output/EUR-11/GERICS/ECMWF-ERAINT/evaluation/r1i1p1/REMO2015/v1/mon/tas/v20180813/"
#prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120"
root_url = f"s3://{bucket}/{prefix}"
group = "CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.mon.v20240920"

# Use ~/.aws/credentials via AWS_PROFILE (defaults to "default") and region
profile = os.environ.get("AWS_PROFILE", "default")
region = os.environ.get("AWS_REGION", "eu-central-1")

# Create a boto3 Session using your local credentials and region
session = Session(profile_name=profile, region_name=region)
credential_provider = Boto3CredentialProvider(session)

# IMPORTANT: include prefix so the store's root IS the Zarr group root
s3_store = S3Store(bucket, skip_signature=True, region="eu-central-1", prefix=prefix)
#s3_store = S3Store("euro-cordex", credential_provider=credential_provider, prefix=prefix)

registry = ObjectStoreRegistry({root_url: s3_store})

#registry=ObjectStoreRegistry({"s3://euro-cordex/": s3_store})
#parser = ZarrParser()
parser = ZarrParser()  # NOTE: current virtualizarr may only support Zarr v2 groups

try:
    vds = open_virtual_dataset(
        url=root_url,
        parser=parser,
        registry=registry,
       # zarr_format=3,
    )
except Exception as e:
    print("open_virtual_dataset failed:", repr(e))
    vds = None

In [4]:
vds

<xarray.Dataset> Size: 291MB
Dimensions:                     (rlat: 412, rlon: 424, time: 407, vertices: 4,
                                 bnds: 2)
Coordinates:
  * rlat                        (rlat) float64 3kB -23.38 -23.27 ... 21.72 21.84
  * rlon                        (rlon) float64 3kB -28.38 -28.27 ... 18.04 18.16
  * time                        (time) datetime64[ns] 3kB 1979-02-15 ... 2012...
    height                      float64 8B ManifestArray<shape=(), dtype=floa...
    lat                         (rlat, rlon) float32 699kB ManifestArray<shap...
    lon                         (rlat, rlon) float32 699kB ManifestArray<shap...
Dimensions without coordinates: vertices, bnds
Data variables:
    lat_vertices                (rlat, rlon, vertices) float32 3MB ManifestAr...
    lon_vertices                (rlat, rlon, vertices) float32 3MB ManifestAr...
    rotated_latitude_longitude  int32 4B ManifestArray<shape=(), dtype=int32,...
    tas                         (time, rlat, rlon) float32 284MB ManifestArra...
    time_bnds                   (time, bnds) int64 7kB ManifestArray<shape=(4...
Attributes: (12/29)
    CORDEX_domain:                  EUR-11
    Conventions:                    CF-1.4
    cmor_version:                   2.9.1
    comment:                        CORDEX Europe RCM REMO 0.11 deg EUR-11.
    contact:                        gerics-cordex@hzg.de
    creation_date:                  2018-06-15T15:20:54Z
    ...                             ...
    realization:                    1
    references:                     http://www.remo-rcm.de/
    source:                         GERICS-REMO2015
    table_id:                       Table mon (Mar 2015) db0b230ff4a2c922671f...
    title:                          GERICS-REMO2015 model output prepared for...
    tracking_id:                    07c8a6b9-424a-4867-815c-5fbce2675d0b

In [ ]:
vds = open_virtual_dataset(
        root_url,
        parser=parser,
        registry=registry,
    )

## Create icechunk repository

In [ ]:
# create on store per dataset

def create_icechunk_repo(key, ds):
    print(f"Adding {key} to icechunk...")
    storage_config = icechunk.s3_storage(
        bucket="euro-cordex",
        prefix=f"CORDEX-CMIP6_icechunk/{key}",
        region='eu-central-1',
    )
    repo = icechunk.Repository.create(storage_config)
    session = repo.writable_session("main")
    to_icechunk(ds, session)
    first_snapshot = session.commit("initial")
    print(f"Finished {key}: {first_snapshot}, {session.status}")

for key, ds in dsets.items():
    create_icechunk_repo(key, ds)

In [ ]:
# create one store with one dataset per group

def create_icechunk_repo_all_in_one(dsets):
    storage_config = icechunk.s3_storage(
        bucket="euro-cordex",
        prefix=f"CORDEX-CMIP6_icechunk_groups",
        region='eu-central-1',
    )
    repo = icechunk.Repository.create(storage_config)
    session = repo.writable_session("main")
    for key, ds in dsets.items():
        print(f"Adding {key} to icechunk...")
        #group = zarr.create_group(session.store, path=key, zarr_format=3)
        ds.load().to_zarr(session.store, group=key, zarr_format=3, consolidated=False)
        #to_icechunk(ds, session, group=key)
    first_snapshot = session.commit("initial")
    print(f"Finished {key}: {first_snapshot}, {session.status}")

create_icechunk_repo_all_in_one(dsets)  # only tas and pr for testing

In [ ]:
storage = icechunk.s3_storage(
    bucket="euro-cordex",
    prefix=f"CORDEX-CMIP6-groups",
    region='eu-central-1',
    )
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")
dt = xr.open_datatree(session.store, consolidated=False, engine="zarr")

In [ ]:
# open all single stores again

for prefix in dsets.keys():
    storage = icechunk.s3_storage(bucket="euro-cordex", prefix=prefix, region="eu-central-1")
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False)

In [ ]:
ds

# VirtualiZarr

In [ ]:
import xarray as xr
from obstore.store import LocalStore
import obstore

from virtualizarr import open_virtual_dataset, open_virtual_mfdataset, open_virtual_dataset
from virtualizarr.parsers import HDFParser, NetCDF3Parser, ZarrParser
from virtualizarr.registry import ObjectStoreRegistry

from pathlib import Path

In [ ]:
import glob

path = "/mnt/CORDEX_CMIP6_tmp/sim_data/CORDEX-CMIP6/DD/EUR-12/GERICS/ERA5/evaluation/r1i1p1f1/REMO2020-2-2/v1-r1/mon/tas/v20241120"
files = sorted(glob.glob(f"{path}/*.nc"))
urls = [f"file://{url}" for url in files]

parser = HDFParser()
store = LocalStore(prefix=path)
registry = ObjectStoreRegistry({url: store for url in urls})

In [ ]:
ds = open_virtual_mfdataset(urls, parser=parser, registry=registry, compat='override', coords="minimal")